In [1]:
import os
import sys
import pandas as pd

In [2]:
# Add the root directory to our system path so we can import from 'src' seamlessly
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.preprocess import clean_data

# Load the raw data from your data/raw directory
# (Since the notebook is inside 'notebooks/', we navigate up '..' and then down into 'data/raw/')
raw_data_path = os.path.join('..', 'data', 'raw', 'diabetic_data.csv')
df_raw_load = pd.read_csv(raw_data_path)

# Test your function!
df_processed = clean_data(df_raw_load)

# Print validation metrics to check if Step 1 works perfectly
print(f"Processed Dataframe Shape: {df_processed.shape}")
print(f"Max Medications Check (Should be capped at 60): {df_processed['num_medications'].max()}")
print(f"Unknown Specialty Count: {df_processed['medical_specialty'].value_counts().get('Unknown', 0)}")

Processed Dataframe Shape: (69883, 45)
Max Medications Check (Should be capped at 60): 60
Unknown Specialty Count: 33561


In [4]:
df_processed.columns

Index(['race', 'gender', 'age', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'medical_specialty', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'target',
       'diag_1_clean', 'diag_2_clean', 'diag_3_clean', 'total_prior_visits'],
      dtype='object')

In [11]:
df_processed['diabetesMed'].value_counts()

diabetesMed
Yes    53222
No     16661
Name: count, dtype: int64

In [12]:
# 1. Define explicit columns lists
numerical_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures', 
    'num_medications', 'number_outpatient', 'number_emergency', 
    'number_inpatient', 'number_diagnoses', 'total_prior_visits'
]

# All columns that contain text labels that need One-Hot Encoding
categorical_cols = [
    'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 
    'admission_source_id', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 
    'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 
    'metformin-rosiglitazone', 'metformin-pioglitazone',
    'diag_1_clean', 'diag_2_clean', 'diag_3_clean'
]

# Columns that can be passed directly into the model once converted to 0 and 1
passthrough_cols = ['change_binary', 'diabetesMed_binary']

# 2. Convert string binary flags into clean 0/1 passthrough integers before building the transformer
df_processed['change_binary'] = (df_processed['change'] == 'Ch').astype(int)
df_processed['diabetesMed_binary'] = (df_processed['diabetesMed'] == 'Yes').astype(int)

# 3. Create our final clean feature matrix X and target array y
# We explicitly select ONLY the features in our defined lists to guarantee no feature leakage
feature_cols = numerical_cols + categorical_cols + passthrough_cols
X = df_processed[feature_cols]
y = df_processed['target']

print(f"Total features selected for modeling: {len(feature_cols)}")
print(f"Target column distribution:\n{y.value_counts()}")

Total features selected for modeling: 44
Target column distribution:
target
0    63638
1     6245
Name: count, dtype: int64


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Define the separate pipeline blocks
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 2. Combine all blocks into a unified ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical_cols),
        ('cat', cat_pipeline, categorical_cols),
        ('pass', 'passthrough', passthrough_cols)
    ],
    remainder='drop' # Automatically drops anything not listed, preventing feature leakage
)

print("ColumnTransformer constructed successfully!")
print(preprocessor)

ColumnTransformer constructed successfully!
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['time_in_hospital', 'num_lab_procedures',
                                  'num_procedures', 'num_medications',
                                  'number_outpatient', 'number_emergency',
                                  'number_inpatient', 'number_diagnoses',
                                  'total_prior_visits']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImpu...
                                  'chlorpropamide', 'glimepiride',
                                  'acetohexamide', 'glipizide', 'glyburide',
                          

In [14]:
from sklearn.model_selection import train_test_split

# Perform the stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

print("Train/Test Split complete!")
print(f"Original Training Features Shape: {X_train.shape}")
print(f"Original Testing Features Shape: {X_test.shape}")

Train/Test Split complete!
Original Training Features Shape: (55906, 44)
Original Testing Features Shape: (13977, 44)


In [15]:
from imblearn.over_sampling import SMOTE

# 1. Fit the transformer on X_train and transform it
X_train_transformed = preprocessor.fit_transform(X_train)

# 2. Transform X_test using the already fitted transformers (No fitting here!)
X_test_transformed = preprocessor.transform(X_test)

# 3. Instantiate SMOTE and resample the training set only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_transformed, y_train)

print("Preprocessing transformations and SMOTE oversampling complete!")

Preprocessing transformations and SMOTE oversampling complete!


In [16]:
print("=== TRAINING SET DISTRIBUTION (AFTER SMOTE) ===")
print("Value Counts:")
print(y_train_resampled.value_counts())
print("\nPercentage Breakdown:")
print(y_train_resampled.value_counts(normalize=True) * 100)
print(f"X_train_resampled shape: {X_train_resampled.shape}")

print("\n")

print("=== TESTING SET DISTRIBUTION (UNTOUCHED REAL-WORLD) ===")
print("Value Counts:")
print(y_test.value_counts())
print("\nPercentage Breakdown:")
print(y_test.value_counts(normalize=True) * 100)
print(f"X_test_transformed shape: {X_test_transformed.shape}")

=== TRAINING SET DISTRIBUTION (AFTER SMOTE) ===
Value Counts:
target
0    50910
1    50910
Name: count, dtype: int64

Percentage Breakdown:
target
0    50.0
1    50.0
Name: proportion, dtype: float64
X_train_resampled shape: (101820, 246)


=== TESTING SET DISTRIBUTION (UNTOUCHED REAL-WORLD) ===
Value Counts:
target
0    12728
1     1249
Name: count, dtype: int64

Percentage Breakdown:
target
0    91.063891
1     8.936109
Name: proportion, dtype: float64
X_test_transformed shape: (13977, 246)
